In [ ]:
import pandas as pd

df = pd.read_csv('../Data/CPREval_Annotation.csv')

selected_columns = ['Additional Label', 'Main Label', 'annotation_id', 'annotator', 'id']
filtered_df = df[selected_columns]

print(filtered_df.head(20))

In [ ]:
import pandas as pd
from scipy.stats import fisher_exact, chi2_contingency
from itertools import product

def preprocess_data(df):
    df['Additional Label'] = df['Additional Label'].str.split(',')
    exploded_df = df.explode('Additional Label')
    
    exploded_df = exploded_df.dropna(subset=['Main Label', 'Additional Label'])
    exploded_df['Main Label'] = exploded_df['Main Label'].str.strip()
    exploded_df['Additional Label'] = exploded_df['Additional Label'].str.strip()
    
    return exploded_df

def significance_testing(exploded_df):
    main_labels = exploded_df['Main Label'].unique()
    additional_labels = exploded_df['Additional Label'].unique()
    
    results = []
    
    for main, add in product(main_labels, additional_labels):
        contingency_table = pd.crosstab(
            exploded_df['Main Label'] == main,
            exploded_df['Additional Label'] == add,
            margins=False
        ).values
        
        if contingency_table.sum() < 5:
            continue
        
        if contingency_table.min() < 5:
            _, p_value = fisher_exact(contingency_table)
            test_type = "Fisher's Exact"
        else:
            _, p_value, _, _ = chi2_contingency(contingency_table)
            test_type = "Chi-square"
        
        co_occurrence = contingency_table[1, 1]
        
        results.append({
            'Main Label': main,
            'Additional Label': add,
            'Co-occurrence': co_occurrence,
            'Test Type': test_type,
            'P-value': p_value
        })
    
    return pd.DataFrame(results)

def calculate_probabilities(exploded_df):
    main_counts = exploded_df['Main Label'].value_counts()
    add_counts = exploded_df['Additional Label'].value_counts()
    
    joint_prob = pd.crosstab(
        exploded_df['Main Label'],
        exploded_df['Additional Label'],
        normalize='all'
    )
    
    cond_prob_main_to_add = joint_prob.div(main_counts, axis=0)
    cond_prob_add_to_main = joint_prob.div(add_counts, axis=1).T
    
    return {
        'P(Additional|Main)': cond_prob_main_to_add,
        'P(Main|Additional)': cond_prob_add_to_main,
        'Joint Probability': joint_prob
    }

def main_analysis(filtered_df):
    processed_df = preprocess_data(filtered_df)
    
    test_results = significance_testing(processed_df)
    
    probabilities = calculate_probabilities(processed_df)
    
    return test_results, probabilities

if __name__ == "__main__":
    test_results, probs = main_analysis(filtered_df)
    
    print("Significance Test Results:")
    print(test_results.sort_values(by='P-value').head(10))
    
    print("\nConditional Probability Examples (P(Additional|Main)):")
    print(probs['P(Additional|Main)'].head())

In [ ]:
def get_top_conditional_pairs(prob_matrix, top_n=10):
    return prob_matrix.stack().reset_index(name='Probability')\
                  .sort_values(by='Probability', ascending=False)\
                  .head(top_n)

top_pairs = get_top_conditional_pairs(probs['P(Additional|Main)'])
print(top_pairs)

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv('./Data/CPREval_Annotation.csv')

selected_columns = ['Additional Label', 'Main Label', 'annotation_id', 'annotator', 'id']
filtered_df = df[selected_columns]

def create_transactions(df):
    transactions = []
    for index, row in df.iterrows():
        main = "M:" + str(row['Main Label']).strip()
        transaction = [main]
        additional = row['Additional Label']
        if pd.notna(additional):
            labels = additional.split(',')
            additional_items = ["A:" + label.strip() for label in labels if label.strip()]
            transaction.extend(additional_items)
        transactions.append(transaction)
    return transactions

transactions = create_transactions(filtered_df)
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = apriori(df_encoded, min_support=0.028, use_colnames=True)
print("\nFrequent itemsets (sorted by support, descending):")
print(frequent_itemsets.sort_values(by='support', ascending=False))

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.2)
print("\nAll association rules (sorted by confidence, descending):")
print(rules.sort_values(by='confidence', ascending=False))

rules_main_to_add = rules[
    rules['antecedents'].apply(lambda s: len(s) == 1 and all(item.startswith("M:") for item in s)) &
    rules['consequents'].apply(lambda s: all(item.startswith("A:") for item in s))
]

print("\nMain Label → Additional Label association rules (sorted by confidence, descending):")
print(rules_main_to_add.sort_values(by='confidence', ascending=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


frequent_itemsets['itemset'] = frequent_itemsets['itemsets'].apply(lambda x: ', '.join(list(x)))
frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False)


plt.figure(figsize=(10, 6))
sns.barplot(x='support', y='itemset', data=frequent_itemsets.head(20), palette='viridis')
plt.title('Top 20 Frequent Itemsets by Support')
plt.xlabel('Support')
plt.ylabel('Itemset')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


alpha = 2.5
x = np.linspace(1, 10, 1000)
y = x ** (-alpha)

plt.figure(figsize=(10, 6))
plt.plot(x, y, label=f'Power-Law Distribution (α={alpha})')
plt.xlabel('x')
plt.ylabel('P(x)')
plt.title('Long-Tail Distribution (Power-Law)')
plt.legend()
plt.grid(True)
plt.show()